In [1]:
import keras

I0000 00:00:1792325677.114668   33894 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792325677.595298   33894 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792325678.858559   33894 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
from keras import ops

In [3]:
from typing import Any


class NaiveDense:
    def __init__(self, input_size, output_size, activation=None):
        self.activation = activation
        self.W = keras.Variable(
            # Creates a matrix W of shape (input_size, output_size),
            # initialized with random values drawn from a uniform
            # distribution
            shape=(input_size, output_size), initializer="uniform"
        )
        # Creates a vector b of shape (output_size,), initialized with
        # zeros
        self.b = keras.Variable(shape=(output_size,), initializer="zeros")
    
    # Applies the forward pass
    def __call__(self, inputs):
        x = ops.matmul(inputs, self.W)
        x = x + self.b
        if self.activation is not None:
            x = self.activation(x)
        return x
    
    @property
    # The convenience method for retrieving the layer's weights
    def weights(self):
        return [self.W, self.b]

In [4]:
class NaiveSequential:
    def __init__(self, layers):
        self.layers = layers

    def __call__(self, inputs):
        x = inputs
        for layer in self.layers:
            x = layer(x)
        return x

    @property
    def weights(self):
        weights = []
        for layer in self.layers:
            weights += layer.weights
        return weights

In [5]:
model = NaiveSequential(
    [
        NaiveDense(input_size=28 * 28, output_size=512, activation=ops.relu),
        NaiveDense(input_size=512, output_size=10, activation=ops.softmax),
    ]
)

E0000 00:00:1792325912.701800   33894 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [7]:
model.weights

[<Variable path=variable, shape=(784, 512), dtype=float32, value=[[-2.4671925e-02 -6.9714710e-04 -1.4063131e-02 ...  3.0657556e-02
   -2.1151453e-04  2.7331259e-02]
  [-4.8899889e-02 -2.8685261e-02  1.7730009e-02 ... -1.7945994e-02
    4.7670826e-03  1.5715849e-02]
  [ 2.5767390e-02  4.1817054e-03  2.2595529e-02 ... -2.6567686e-02
    1.1207759e-02 -1.7569195e-02]
  ...
  [ 4.4819426e-02  3.8372267e-02  2.0666931e-02 ...  4.2993177e-02
   -2.2143126e-02  7.5519085e-05]
  [ 1.4704298e-02 -1.8630661e-02  4.2933475e-02 ...  1.0606170e-02
    4.4671204e-02  4.2597499e-02]
  [ 4.2203180e-03  4.3534134e-02  1.8465962e-02 ...  3.7989441e-02
    2.0284902e-02 -1.6796686e-02]]>,
 <Variable path=variable_1, shape=(512,), dtype=float32, value=[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.

In [8]:
assert len(model.weights) == 4